# Partial Fine-Tuning Model Comparison

Train ChestMNIST classifiers by unfreezing the final 1-5 transformer blocks of
each DINO-style backbone. Every block-count/LR/weight-decay configuration is first
trained through a short search schedule. The selected configuration is then
initialized from scratch and trained through the full checkpoint schedule.

This notebook always performs the image-based training work it measures. It does
not reuse frozen feature banks. Search cost, final-run cost, and total experimental
cost are saved separately.

In [1]:
import os

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import gc
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import transformers
from packaging.version import Version
from tqdm.notebook import tqdm

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

MIN_TRANSFORMERS_FOR_DINOV3 = Version("4.57.2")
if Version(transformers.__version__) < MIN_TRANSFORMERS_FOR_DINOV3:
    raise RuntimeError(
        "DINOv3 requires transformers>=4.57.2,<5 in this project. "
        f"Current transformers={transformers.__version__}."
    )

from src.data import get_small_data
from src.results import make_run_dir, save_partial_finetune_outputs, validate_save_request
from src.train import make_partial_finetune_grid, run_partial_finetune_grid

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Project root:", project_root)
print("Device:", device)
print("Transformers:", transformers.__version__)

Project root: c:\Users\vasko\Downloads\xai\project
Device: cuda
Transformers: 4.57.6


## Config

In [ ]:
dataset_name = "chestmnist"
data_fraction = 1.0
dataset_seed = 0
save_outputs = True

# Search every configuration briefly, then fully retrain the selected configuration.
search_checkpoint_epochs = [1, 3, 5]

# Use a compact final-run checkpoint and threshold schedule.
checkpoint_epochs = [1, 2, 3, 5, 7, 10, 15, 20, 30]
thresholds = [0.05, 0.1, 0.2, 0.3, 0.5]
selection_metric = "mean_auc"

# Block-count-controlled partial fine-tuning: the same block counts for every model.
train_grid = make_partial_finetune_grid(
    num_unfrozen_blocks=[1, 2, 3],
    backbone_lrs=[1e-6, 3e-6],
    weight_decays=[1e-4],
    head_lr_multiplier=10.0,
)

use_amp = True
gradient_checkpointing = True
output_root = project_root / "outputs"

# Image-based fine-tuning needs smaller batches than frozen feature extraction.
model_configs = [
    {
        "run_name": "dinov2_small_224",
        "model_name": "facebook/dinov2-small",
        "image_size": 224,
        "batch_size": 16,
    },
    {
        "run_name": "dinov2_large_224",
        "model_name": "facebook/dinov2-large",
        "image_size": 224,
        "batch_size": 4,
    },
    {
        "run_name": "dinov3_large_224",
        "model_name": "facebook/dinov3-vitl16-pretrain-lvd1689m",
        "image_size": 224,
        "batch_size": 4,
    },
    {
        "run_name": "stanford_dinov2_xray_224",
        "model_name": "StanfordAIMI/dinov2-base-xray-224",
        "image_size": 224,
        "batch_size": 8,
    },
    {
        "run_name": "rad_dino_518",
        "model_name": "microsoft/rad-dino",
        "image_size": 518,
        "batch_size": 2,
    },
]

validate_save_request(save_outputs, data_fraction)
models_to_run = model_configs
print("search configurations per model:", len(train_grid))
print("search epochs per configuration:", max(search_checkpoint_epochs))
print("final retraining epochs:", max(checkpoint_epochs))
print("search checkpoints per configuration:", len(search_checkpoint_epochs))
print("final-run checkpoints:", len(checkpoint_epochs))
print("thresholds per checkpoint:", len(thresholds))
print(
    "metric rows per model:",
    (
        len(train_grid) * len(search_checkpoint_epochs)
        + len(checkpoint_epochs)
    )
    * len(thresholds),
)
models_to_run

search configurations per model: 4
search epochs per configuration: 5
final retraining epochs: 30
search checkpoints per configuration: 3
final-run checkpoints: 9
thresholds per checkpoint: 5
metric rows per model: 105


[{'run_name': 'dinov2_small_224',
  'model_name': 'facebook/dinov2-small',
  'image_size': 224,
  'batch_size': 16},
 {'run_name': 'dinov2_large_224',
  'model_name': 'facebook/dinov2-large',
  'image_size': 224,
  'batch_size': 4},
 {'run_name': 'dinov3_large_224',
  'model_name': 'facebook/dinov3-vitl16-pretrain-lvd1689m',
  'image_size': 224,
  'batch_size': 4},
 {'run_name': 'stanford_dinov2_xray_224',
  'model_name': 'StanfordAIMI/dinov2-base-xray-224',
  'image_size': 224,
  'batch_size': 8},
 {'run_name': 'rad_dino_518',
  'model_name': 'microsoft/rad-dino',
  'image_size': 518,
  'batch_size': 2}]

## Run Experiments

Runtime accounting:

- The comparison-facing `*_grid_seconds` fields count search plus final retraining,
  consistent with the total-work accounting used by the kNN and linear-probe runs.
- `search_*_seconds` and `final_*_seconds` preserve the two stages separately.
- Validation inference runs only at requested checkpoints.
- Threshold scoring reuses checkpoint predictions and does not retrain models.
- Every completed config updates `_progress/partial_finetune_results.csv`,
  `_progress/current_best.csv`, and the cross-model live results table.
- Compact best checkpoints save trainable weights and validation predictions only;
  completed configs are skipped automatically when this cell is rerun.

In [3]:
all_histories = []
all_trials = []
all_summaries = []
all_per_class = []
all_metadata = []

for config in tqdm(models_to_run, desc="Models", leave=True):
    run_start = time.perf_counter()
    run_name = config["run_name"]
    model_name = config["model_name"]
    image_size = config["image_size"]
    batch_size = config["batch_size"]

    print("\n" + "=" * 80)
    print(run_name)
    print(model_name)

    context = {
        "dataset_name": dataset_name,
        "adaptation_method": "partial_finetune",
        "run_name": run_name,
        "model_name": model_name,
        "image_size": image_size,
        "batch_size": batch_size,
        "data_fraction": data_fraction,
        "dataset_seed": dataset_seed,
    }

    data_start = time.perf_counter()
    train_loader, val_loader, class_names = get_small_data(
        batch_size=batch_size,
        image_size=image_size,
        data_fraction=data_fraction,
        seed=dataset_seed,
    )
    data_load_seconds = time.perf_counter() - data_start

    result_run_dir = make_run_dir(
        output_root,
        dataset_name,
        f"partial_finetuning/{run_name}",
        data_fraction=data_fraction,
        dataset_seed=dataset_seed,
    )

    result = run_partial_finetune_grid(
        model_name=model_name,
        num_classes=len(class_names),
        train_loader=train_loader,
        val_loader=val_loader,
        device=device,
        class_names=class_names,
        train_grid=train_grid,
        checkpoint_epochs=checkpoint_epochs,
        search_checkpoint_epochs=search_checkpoint_epochs,
        thresholds=thresholds,
        selection_metric=selection_metric,
        seed=dataset_seed,
        context=context,
        use_amp=use_amp,
        gradient_checkpointing=gradient_checkpointing,
        show_progress=True,
        progress_run_dir=result_run_dir if save_outputs else None,
    )

    model_init_seconds = float(result["metadata"]["model_init_grid_seconds"])
    partial_train_seconds = float(result["metadata"]["partial_train_grid_seconds"])
    val_eval_seconds = float(result["metadata"]["val_eval_grid_seconds"])
    threshold_search_seconds = float(result["metadata"]["threshold_search_grid_seconds"])
    adaptation_seconds = float(model_init_seconds + partial_train_seconds)
    efficiency_counted_seconds = float(
        adaptation_seconds + val_eval_seconds + threshold_search_seconds
    )
    actual_wall_seconds = float(time.perf_counter() - run_start)

    metadata = {
        **result["metadata"],
        "data_load_seconds": float(data_load_seconds),
        "feature_extraction_seconds": 0.0,
        "adaptation_seconds": adaptation_seconds,
        "inference_or_eval_seconds": val_eval_seconds,
        "threshold_search_seconds": threshold_search_seconds,
        "efficiency_counted_seconds": efficiency_counted_seconds,
        "actual_wall_seconds": actual_wall_seconds,
        "shortcut_used": False,
    }

    if save_outputs:
        save_partial_finetune_outputs(
            result_run_dir,
            history_df=result["history"],
            summary_df=result["summary"],
            per_class_df=result["per_class"],
            trials_df=result["trials"],
            metadata=metadata,
        )

    all_histories.append(result["history"])
    all_trials.append(result["trials"])
    all_summaries.append(result["summary"])
    all_per_class.append(result["per_class"])
    all_metadata.append(metadata)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

combined_history_df = pd.concat(all_histories, ignore_index=True)
combined_trials_df = pd.concat(all_trials, ignore_index=True)
selected_summary_df = pd.concat(all_summaries, ignore_index=True)
selected_per_class_df = pd.concat(all_per_class, ignore_index=True)
metadata_df = pd.DataFrame(all_metadata)

selected_summary_df

Models:   0%|          | 0/5 [00:00<?, ?it/s]


dinov2_small_224
facebook/dinov2-small
Using downloaded and verified file: C:\Users\vasko\.medmnist\chestmnist.npz
Using downloaded and verified file: C:\Users\vasko\.medmnist\chestmnist.npz


Partial fine-tune search configs:   0%|          | 0/4 [00:00<?, ?it/s]

Current partial fine-tune:   0%|          | 0/5 [00:00<?, ?it/s]

c:\Users\vasko\anaconda3\envs\cumid\Lib\site-packages\transformers\integrations\sdpa_attention.py:96: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


KeyboardInterrupt: 

## Reporting Policy

Use one row from the freshly retrained final model as the headline comparison.
Mean AUROC selects configurations/checkpoints; macro F1 breaks AUROC ties to choose
the operating threshold. Report the selected block count, epoch, learning rates,
weight decay,
threshold, trainable parameters, runtime, and GPU memory. Search-grid distributions
remain robustness/audit evidence rather than headline averages.

In [ ]:
display_names = {
    "dinov2_small_224": "DINOv2-S 224",
    "dinov2_large_224": "DINOv2-L 224",
    "dinov3_large_224": "DINOv3-L 224",
    "stanford_dinov2_xray_224": "Stanford X-ray DINOv2-B 224",
    "rad_dino_518": "RAD-DINO 518",
}
model_groups = {
    "dinov2_small_224": "Generic DINOv2",
    "dinov2_large_224": "Generic DINOv2",
    "dinov3_large_224": "Generic DINOv3",
    "stanford_dinov2_xray_224": "Medical/X-ray",
    "rad_dino_518": "Medical/X-ray",
}
group_colors = {
    "Generic DINOv2": "tab:blue",
    "Generic DINOv3": "tab:purple",
    "Medical/X-ray": "tab:green",
}

plot_summary_df = selected_summary_df.copy()
plot_summary_df["display_name"] = plot_summary_df["run_name"].map(display_names)
plot_summary_df["model_group"] = plot_summary_df["run_name"].map(model_groups)

plot_trials_df = combined_trials_df.copy()
plot_trials_df["display_name"] = plot_trials_df["run_name"].map(display_names)
plot_trials_df["model_group"] = plot_trials_df["run_name"].map(model_groups)
search_trials_df = plot_trials_df[plot_trials_df["stage"] == "search"].copy()
final_trials_df = plot_trials_df[plot_trials_df["stage"] == "final"].copy()

plot_metadata_df = metadata_df.copy()
plot_metadata_df["display_name"] = plot_metadata_df["run_name"].map(display_names)
plot_metadata_df["model_group"] = plot_metadata_df["run_name"].map(model_groups)

plot_per_class_df = selected_per_class_df.copy()
plot_per_class_df["display_name"] = plot_per_class_df["run_name"].map(display_names)
plot_per_class_df["model_group"] = plot_per_class_df["run_name"].map(model_groups)

### Checkpoint Curves by Model

In [ ]:
checkpoint_best_df = (
    final_trials_df.sort_values(
        ["run_name", "train_trial_id", "epoch", selection_metric],
        ascending=[True, True, True, False],
    )
    .groupby(["run_name", "train_trial_id", "epoch"], as_index=False)
    .head(1)
)

checkpoint_curve_df = (
    checkpoint_best_df.groupby(["display_name", "model_group", "epoch"], sort=False)
    .agg(
        runs=("train_trial_id", "nunique"),
        mean_auc_mean=("mean_auc", "mean"),
        mean_auc_std=("mean_auc", "std"),
        f1_macro_mean=("f1_macro", "mean"),
        f1_macro_std=("f1_macro", "std"),
        f1_micro_mean=("f1_micro", "mean"),
        f1_micro_std=("f1_micro", "std"),
        mean_accuracy_mean=("mean_accuracy", "mean"),
        mean_accuracy_std=("mean_accuracy", "std"),
        exact_match_accuracy_mean=("exact_match_accuracy", "mean"),
        exact_match_accuracy_std=("exact_match_accuracy", "std"),
    )
    .reset_index()
)

metrics_to_plot = [
    ("mean_auc", "Mean AUROC"),
    ("f1_macro", "Macro F1"),
    ("f1_micro", "Micro F1"),
    ("mean_accuracy", "Mean Accuracy"),
    ("exact_match_accuracy", "Exact Match Accuracy"),
]
fig, axes = plt.subplots(2, 3, figsize=(17, 8))
axes = axes.ravel()
for ax, (metric, title) in zip(axes, metrics_to_plot):
    for display_name, group_df in checkpoint_curve_df.groupby("display_name", sort=False):
        group_df = group_df.sort_values("epoch")
        ax.plot(group_df["epoch"], group_df[f"{metric}_mean"], marker="o", label=display_name)
    ax.set_title(title)
    ax.set_xlabel("Training epoch checkpoint")
    ax.set_ylabel(title)
axes[-1].axis("off")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2)
fig.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

### Selected Metrics and Efficiency

In [ ]:
full_plot_df = plot_summary_df.sort_values("mean_auc", ascending=False).copy()
model_order = full_plot_df["display_name"].tolist()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric, title in [
    (axes[0], "mean_auc", "Mean AUROC"),
    (axes[1], "f1_macro", "Macro F1"),
    (axes[2], "f1_micro", "Micro F1"),
]:
    colors = [group_colors[group] for group in full_plot_df["model_group"]]
    ax.bar(full_plot_df["display_name"], full_plot_df[metric], color=colors)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

efficiency_df = full_plot_df.merge(
    plot_metadata_df[
        [
            "run_name",
            "efficiency_counted_seconds",
            "actual_wall_seconds",
            "peak_gpu_memory_mb",
            "adaptation_seconds",
            "inference_or_eval_seconds",
            "threshold_search_seconds",
            "search_counted_seconds",
            "final_counted_seconds",
            "shortcut_used",
        ]
    ],
    on="run_name",
    how="left",
)

fig, axes = plt.subplots(2, 3, figsize=(17, 9), sharex="col")
scatter_specs = [
    ("trainable_params", "Trainable Parameters", True),
    ("efficiency_counted_seconds", "Counted Method Seconds", False),
    ("peak_gpu_memory_mb", "Peak GPU Memory MB", False),
]
metric_specs = [("mean_auc", "Mean AUROC"), ("f1_macro", "Macro F1")]
for row_idx, (metric, metric_title) in enumerate(metric_specs):
    for ax, (x_col, x_title, log_x) in zip(axes[row_idx], scatter_specs):
        for _, row in efficiency_df.iterrows():
            ax.scatter(
                row[x_col],
                row[metric],
                s=90,
                color=group_colors[row["model_group"]],
                edgecolor="black",
            )
            ax.annotate(row["display_name"], (row[x_col], row[metric]), xytext=(5, 5), textcoords="offset points", fontsize=8)
        if log_x:
            ax.set_xscale("log")
        ax.set_title(f"{metric_title} vs {x_title}")
        ax.set_xlabel(x_title)
        ax.set_ylabel(metric_title)
plt.tight_layout()
plt.show()

### Report Table

In [ ]:
report_table_df = full_plot_df.merge(
    plot_metadata_df[
        [
            "run_name",
            "selection_metric",
            "num_train_trials",
            "num_epoch_checkpoints",
            "num_thresholds",
            "selection_protocol",
            "num_search_train_trials",
            "num_final_train_trials",
            "num_search_epoch_checkpoints",
            "num_final_epoch_checkpoints",
            "adaptation_seconds",
            "inference_or_eval_seconds",
            "threshold_search_seconds",
            "search_counted_seconds",
            "final_counted_seconds",
            "efficiency_counted_seconds",
            "actual_wall_seconds",
            "peak_gpu_memory_mb",
            "shortcut_used",
        ]
    ],
    on="run_name",
    how="left",
)

report_cols = [
    "display_name",
    "mean_auc",
    "f1_macro",
    "f1_micro",
    "mean_accuracy",
    "exact_match_accuracy",
    "num_unfrozen_blocks",
    "epoch",
    "backbone_lr",
    "head_lr",
    "weight_decay",
    "threshold",
    "trainable_params",
    "trainable_param_fraction",
    "search_counted_seconds",
    "final_counted_seconds",
    "efficiency_counted_seconds",
    "actual_wall_seconds",
    "peak_gpu_memory_mb",
    "shortcut_used",
]
report_table_df[report_cols].sort_values(selection_metric, ascending=False)

### Grid Robustness and Per-Class Diagnostics

In [ ]:
best_threshold_per_checkpoint_df = (
    search_trials_df.sort_values(
        ["run_name", "train_trial_id", "epoch", selection_metric],
        ascending=[True, True, True, False],
    )
    .groupby(["run_name", "train_trial_id", "epoch"], as_index=False)
    .head(1)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
best_threshold_per_checkpoint_df.boxplot(column=selection_metric, by="display_name", ax=axes[0], rot=45)
best_threshold_per_checkpoint_df.boxplot(column="mean_auc", by="display_name", ax=axes[1], rot=45)
axes[0].set_title(f"{selection_metric} across configs/checkpoints")
axes[1].set_title("AUROC across configs/checkpoints")
plt.suptitle("")
plt.tight_layout()
plt.show()

per_class_table_df = plot_per_class_df[
    ["display_name", "class_name", "true_positive_rate", "predicted_positive_rate", "accuracy", "f1", "auc"]
].sort_values(["display_name", "class_name"])
per_class_table_df

In [ ]:
selected_train_ids = full_plot_df[["run_name", "train_trial_id"]].drop_duplicates()
selected_history_df = combined_history_df.merge(
    selected_train_ids,
    on=["run_name", "train_trial_id"],
    how="inner",
)
selected_history_df = selected_history_df[selected_history_df["is_checkpoint"]].copy()
selected_history_df["display_name"] = selected_history_df["run_name"].map(display_names)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for display_name, group in selected_history_df.groupby("display_name", sort=False):
    axes[0].plot(group["epoch"], group["val_mean_auc"], marker="o", label=display_name)
    axes[1].plot(group["epoch"], group["val_f1_macro"], marker="o", label=display_name)
axes[0].set_title("Selected configuration validation AUROC")
axes[1].set_title("Selected configuration validation macro F1 at 0.5")
axes[0].set_xlabel("Epoch")
axes[1].set_xlabel("Epoch")
axes[0].set_ylabel("Mean AUROC")
axes[1].set_ylabel("Macro F1")
axes[1].legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()